# Efbet Scraper (Selenium)
**Note:** Efbet is a Single Page Application (SPA). Selectors (class names) may change frequently. If the script fails, inspect the page elements and update the `By.CSS_SELECTOR` or `By.XPATH` values.

In [ ]:
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
import pandas as pd
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

In [ ]:
# --- SETUP DRIVER ---
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
# options.add_argument("--headless") # Uncomment to run in background

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
wait = WebDriverWait(driver, 15)

In [ ]:
# --- HELPER FUNCTIONS ---

def accept_cookies():
    """Checks for cookie banner and accepts if present."""
    try:
        # Generic XPath to find 'Accept' or 'Agree' buttons
        cookie_btn = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Accept') or contains(text(), 'Agree')]")))
        cookie_btn.click()
        print("Cookies accepted.")
        time.sleep(1)
    except TimeoutException:
        print("No cookie banner found (or already accepted).")

def scroll_page():
    """Scrolls to the bottom of the page to trigger lazy loading."""
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3) # Wait for DOM to update
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height
    print("Scrolling complete.")

def apply_24h_filter():
    """Attempts to click the '24h' or 'Today' filter."""
    try:
        # Try multiple XPaths for the time filter
        filter_btn = wait.until(EC.element_to_be_clickable((By.XPATH, "//span[contains(text(), '24h') or contains(text(), 'Today')]")))
        filter_btn.click()
        print("Applied 24h filter.")
        time.sleep(3)
    except TimeoutException:
        print("24h filter not found or not clickable. Proceeding with default view.")

## 1. Football

In [ ]:
# 1. Navigate to Football
driver.get('https://www.efbet.gr/en/') # Base URL
time.sleep(3)
accept_cookies()

try:
    # Navigate via sidebar if direct URL doesn't work
    sport_link = wait.until(EC.element_to_be_clickable((By.XPATH, "//span[contains(text(), 'Soccer') or contains(text(), 'Football')]")))
    sport_link.click()
    print("Navigated to Football.")
except TimeoutException:
    print("Could not navigate to Football via menu. Trying direct URL logic if available.")

# 2. Apply Filters and Scroll
apply_24h_filter()
scroll_page()

# 3. Extract Raw Text
try:
    # 'center-view-content' is a common class for the middle grid in betting SPAs. Adjust if needed.
    main_container = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.center-view-content, div.main-content, div#center")))
    football_string = main_container.text
    print(f"Extracted {len(football_string)} characters.")
except Exception as e:
    print(f"Error extracting text: {e}")
    football_string = ""

In [ ]:
# 4. Process Data
initial_list = football_string.split('\n')

# Filter Garbage
remove_elements = ['Soccer', 'All', '1', 'X', '2', 'Over', 'Under', 'BTS', 'Double Chance', 'Draw No Bet', 'Handicap']
list_1 = [x for x in initial_list if x not in remove_elements and len(x) > 1]

# Identify Match Start Indices (looking for time pattern like 14:00)
# Note: You might need to adjust ':' check if date format differs (e.g. 24/03)
index_match = [i for i, x in enumerate(list_1) if ':' in x and len(x) < 6 and x[0].isdigit()]

if index_match:
    sublists_matches = [list_1[i:j] for i, j in zip(index_match, index_match[1:] + [len(list_1)])]

    # Standardize Row Length (Adjust '15' based on actual columns found)
    target_len = 15
    for sublist in sublists_matches:
        if len(sublist) < target_len:
            sublist.extend(['No_bet'] * (target_len - len(sublist)))
        elif len(sublist) > target_len:
            del sublist[target_len:]

    columns = ['time', 'team1', 'team2', '1', 'X', '2', 'O_odds', 'U_odds', 'misc1', 'misc2', 'misc3', 'misc4', 'misc5', 'misc6', 'misc7']
    df_football = pd.DataFrame(sublists_matches, columns=columns)
else:
    print("No match patterns found. Check 'list_1' content.")
    df_football = pd.DataFrame()

In [ ]:
df_football.head()

## 2. Basketball

In [ ]:
# 1. Navigate
driver.get('https://www.efbet.gr/en/') 
time.sleep(3)

try:
    sport_link = wait.until(EC.element_to_be_clickable((By.XPATH, "//span[contains(text(), 'Basketball')]")))
    sport_link.click()
    print("Navigated to Basketball.")
except TimeoutException:
    print("Navigation failed.")

# 2. Filter & Scroll
apply_24h_filter()
scroll_page()

# 3. Extract
try:
    main_container = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.center-view-content, div.main-content")))
    basketball_string = main_container.text
except:
    basketball_string = ""

In [ ]:
# 4. Process Data
initial_list = basketball_string.split('\n')
remove_elements = ['Basketball', 'Winner', 'Handicap', 'Over/Under', 'Total', '1', '2']
list_b = [x for x in initial_list if x not in remove_elements and len(x) > 1]

index_match = [i for i, x in enumerate(list_b) if ':' in x and len(x) < 6 and x[0].isdigit()]

if index_match:
    sublists_matches = [list_b[i:j] for i, j in zip(index_match, index_match[1:] + [len(list_b)])]
    
    target_len = 12
    for sublist in sublists_matches:
        if len(sublist) < target_len:
            sublist.extend(['No_bet'] * (target_len - len(sublist)))
        del sublist[target_len:]

    columns = ['time', 'team1', 'team2', 'win1', 'win2', 'handicap_val', 'h1', 'h2', 'total_val', 'over', 'under', 'misc']
    df_basketball = pd.DataFrame(sublists_matches, columns=columns)
else:
    df_basketball = pd.DataFrame()

In [ ]:
df_basketball.head()

## 3. Tennis

In [ ]:
# 1. Navigate
driver.get('https://www.efbet.gr/en/')
time.sleep(3)

try:
    sport_link = wait.until(EC.element_to_be_clickable((By.XPATH, "//span[contains(text(), 'Tennis')]")))
    sport_link.click()
    print("Navigated to Tennis.")
except TimeoutException:
    print("Navigation failed.")

# 2. Filter & Scroll
apply_24h_filter()
scroll_page()

# 3. Extract
try:
    main_container = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.center-view-content, div.main-content")))
    tennis_string = main_container.text
except:
    tennis_string = ""

In [ ]:
# 4. Process Data
initial_list = tennis_string.split('\n')
remove_elements = ['Tennis', 'Winner', 'Set Betting', 'Game Handicap', '1', '2']
list_t = [x for x in initial_list if x not in remove_elements and len(x) > 1]

index_match = [i for i, x in enumerate(list_t) if ':' in x and len(x) < 6 and x[0].isdigit()]

if index_match:
    sublists_matches = [list_t[i:j] for i, j in zip(index_match, index_match[1:] + [len(list_t)])]
    
    target_len = 10
    for sublist in sublists_matches:
        if len(sublist) < target_len:
            sublist.extend(['No_bet'] * (target_len - len(sublist)))
        del sublist[target_len:]

    columns = ['time', 'player1', 'player2', 'win1', 'win2', 'set1', 'set2', 'games_o', 'games_u', 'misc']
    df_tennis = pd.DataFrame(sublists_matches, columns=columns)
else:
    df_tennis = pd.DataFrame()

In [ ]:
df_tennis.head()

In [ ]:
# Cleanup
driver.quit()